In [ ]:
# imports

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean

In [ ]:
# def compute_distance_matrix


def compute_distance_matrix(matrix, distance_func):
    """
    Computes a distance matrix for the given matrix using a specified distance function.

    Parameters:
    - matrix (2D array): Input matrix (shape = (m, n)), where each row represents a vector.
    - distance_func (function): A function that computes the distance between two vectors.

    Returns:
    - distance_matrix (2D array): Distance matrix (shape = (m, m)).
    """
    m = matrix.shape[0]  # Number of rows (vectors)
    distance_matrix = np.zeros((m, m))  # Initialize distance matrix

    for i in range(m):
        for j in range(i + 1, m):
            distance = distance_func(matrix[i, :], matrix[j, :])  # Apply distance function
            distance_matrix[i, j] = distance
            distance_matrix[j, i] = distance  # Symmetric matrix

    return distance_matrix

In [ ]:
# def visualize_2d_vectors


def visualize_2d_vectors(matrix, title="2D Vector Visualization", figure_size=(8, 6)):
    """
    Plots a scatter plot of a 2D matrix where each row is a vector.

    Parameters:
    - matrix (2D array): The data matrix (shape = (m, 2)), where each row is a 2D vector.
    - title (str, optional): Title of the plot. Defaults to "2D Vector Visualization".
    - figure_size (tuple, optional): Figure size. Defaults to (8, 6).

    Returns:
    - None
    """

    plt.figure(figsize=figure_size)
    plt.scatter(matrix[:, 0], matrix[:, 1])

    plt.xlabel("X-axis")
    plt.ylabel("Y-axis")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# def vat_reorder


def vat_reorder(distance_matrix):
    """
    Reorders a given distance matrix using the VAT method.

    Parameters:
    - distance_matrix (2D ndarray): Pairwise distance matrix.

    Returns:
    - vat_matrix (2D ndarray): Reordered distance matrix for cluster visualization.
    - reorder (list): The reordering of indices applied to the distance matrix.
    """
    n = distance_matrix.shape[0]

    # Step 1: Find the most dissimilar pair (farthest apart)
    i, j = np.unravel_index(np.argmax(distance_matrix, axis=None), distance_matrix.shape)

    # Step 2: Start with one of the most distant points
    reorder = [i]
    remaining = set(range(n)) - {i}  # Remaining points to process

    # Step 3: Iteratively find the next closest point to any selected point
    while remaining:
        next_index = min(remaining, key=lambda x: min(distance_matrix[x, p] for p in reorder))
        reorder.append(next_index)
        remaining.remove(next_index)

    # Step 4: Apply the reordering to the distance matrix
    vat_matrix = distance_matrix[np.ix_(reorder, reorder)]

    return vat_matrix, reorder

In [ ]:
# def iVAT_transform


def iVAT_transform(vat_matrix):
    """
    Compute the iVAT distance transform (path-based distances) from a VAT-reordered matrix.

    The input vat_matrix should already be reordered via VAT, meaning its rows and columns
    follow a minimum spanning tree-like sequence. This function replaces each direct distance
    with the “bottleneck” distance along the MST path—i.e., the maximum edge on that path.

    Algorithm steps:
    For each new row r (1..N-1):
        - Find j < r that yields the minimum distance in vat_matrix[r, :r].
          (This j is the MST edge that would connect r to the existing tree.)
        - Set iVat[r, j] to that distance (symmetric as well).
        - For each c < r, c != j:
          iVat[r, c] = max( vat_matrix[r, j], iVat[j, c] )
          (The path from r to c goes through j, so the cost is the maximum of
          the r->j edge and the already computed j->c path.)

    Parameters
    ----------
    vat_matrix : ndarray of shape (N, N)
        The VAT-reordered distance matrix, typically symmetrical and
        with zeros on the diagonal.

    Returns
    -------
    ivat_matrix : ndarray of shape (N, N)
        The iVAT-transformed distance matrix in the same VAT ordering.
    """
    N = vat_matrix.shape[0]
    ivat_matrix = np.zeros_like(vat_matrix)

    for r in range(1, N):
        # Find the closest connection j for row r among previously processed rows
        j = np.argmin(vat_matrix[r, :r])

        # Direct MST edge between r and j
        ivat_matrix[r, j] = vat_matrix[r, j]
        ivat_matrix[j, r] = ivat_matrix[r, j]

        # Update path-based distances to all other previous vertices c
        for c in range(r):
            if c != j:
                # Bottleneck distance: the worst edge on the path r->j->c
                ivat_matrix[r, c] = max(vat_matrix[r, j], ivat_matrix[j, c])
                ivat_matrix[c, r] = ivat_matrix[r, c]

    return ivat_matrix

In [ ]:
# def visualize_vat_reordering


def visualize_matrix(
    matrix,
    labels=None,
    title="Matrix",
    cmap="viridis",
    fontsize=10,
    colorbar_min=None,
    colorbar_max=None,
    figure_size=(8, 6),
):
    """
    Plots a given matrix as a heatmap with cell values and x-axis labels on top.

    Parameters:
    - matrix (2D array): The matrix to visualize (shape = (n, n)).
    - labels (array-like, optional): Labels for the rows and columns (should be a list). Defaults to None.
    - title (str, optional): Title of the plot. Defaults to "Matrix".
    - cmap (str, optional): Colormap for the heatmap. Defaults to "viridis".
    - fontsize (int, optional): Font size for text in the plot. Defaults to 10.
    - colorbar_min (float, optional): Minimum value for the color scale. Defaults to None (auto).
    - colorbar_max (float, optional): Maximum value for the color scale. Defaults to None (auto).
    - figure_size (tuple, optional): Figure size. Defaults to (8, 6).

    Returns:
    - None
    """

    plt.figure(figsize=figure_size)
    ax = plt.gca()  # Get current axis

    # Define color range limits
    vmin = colorbar_min if colorbar_min is not None else np.min(matrix)
    vmax = colorbar_max if colorbar_max is not None else np.max(matrix)

    # Plot heatmap with custom color range
    img = plt.imshow(matrix, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(img)
    cbar.set_label("Value", fontsize=fontsize)

    # Ensure labels are a list of strings
    if labels is not None and len(labels) > 0:
        labels = [str(label) for label in labels]  # Convert to strings if needed
        ax.set_xticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, fontsize=fontsize)
        ax.set_yticks(np.arange(len(labels)))
        ax.set_yticklabels(labels, fontsize=fontsize)
    else:
        n = matrix.shape[0]
        ax.set_xticks(np.arange(n))
        ax.set_xticklabels([f"Row {i + 1}" for i in range(n)], fontsize=fontsize)
        ax.set_yticks(np.arange(n))
        ax.set_yticklabels([f"Column {i + 1}" for i in range(n)], fontsize=fontsize)

    # Move x-axis labels to the top
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()

    # Add cell values
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            plt.text(
                j,
                i,
                f"{matrix[i, j]:.1f}",
                ha="center",
                va="center",
                fontsize=fontsize - 2,
                color="black" if matrix[i, j] > (vmax / 2) else "white",
            )

    # Set title
    plt.title(title, fontsize=fontsize + 2, pad=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# --------------------------------------------------------------------------------------------------
# Generate and visualize 15 vectors from 3 clusters
# --------------------------------------------------------------------------------------------------
np.random.seed(42)  # For reproducibility

m, n = 15, 2  # 15 vectors, each with 2 dimensions
num_clusters = 3
cluster_sizes = [m // num_clusters] * num_clusters  # Equal distribution of points
centers = np.array([[0, 0], [5, 5], [-5, 5]])  # Define cluster centers

# Generate points around each cluster center
data_matrix = np.vstack(
    [centers[i] + np.random.randn(cluster_sizes[i], n) for i in range(num_clusters)]
)
np.random.shuffle(data_matrix)

# Generate random cluster labels for visualization (optional)
cluster_labels = np.repeat(
    np.arange(num_clusters), cluster_sizes[0]
)  # 0, 0, ..., 1, 1, ..., 2, 2, ...

# Call the function
visualize_2d_vectors(data_matrix, title="Generated 2D Clusters")

# --------------------------------------------------------------------------------------------------
# Compute and visualize distance matrix
# --------------------------------------------------------------------------------------------------
distance_matrix = compute_distance_matrix(
    data_matrix, euclidean
)  # Calculate distance(dissimilarity) using Euclidean distance

visualize_matrix(
    distance_matrix,
    labels=["Vec " + str(i + 1) for i in range(m)],
    title="Original Distance Matrix",
    figure_size=(10, 8),
)

# --------------------------------------------------------------------------------------------------
# Compute and visualize VAT-transformed distance matrix
# --------------------------------------------------------------------------------------------------
vat_distance_matrix, reorder = vat_reorder(distance_matrix)

visualize_matrix(
    vat_distance_matrix,
    labels=["Vec " + str(i + 1) for i in range(m)],
    title="VAT Distance Matrix",
    figure_size=(10, 8),
)

# --------------------------------------------------------------------------------------------------
# Compute and visualize iVAT-transformed distance matrix
# --------------------------------------------------------------------------------------------------
iVAT_distance_matrix = iVAT_transform(vat_distance_matrix)

visualize_matrix(
    iVAT_distance_matrix,
    labels=["Vec " + str(i + 1) for i in range(m)],
    title="iVAT Distance Matrix",
    figure_size=(10, 8),
)